# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR⁲](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant-python/) library.

### Dataset Source
The dataset is provided as a [Croissant](https://mlcommons.github.io/croissant/) schema JSON-LD file at:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Install mlcroissant
!pip install mlcroissant

## 1. Data Loading
We load the dataset and its metadata using `mlcroissant`.

The Croissant schema URL is assigned to a variable for easy reuse.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

# Print name and description
print(f"{meta.name}:\n{meta.description}\n")
print(f"Version: {meta.version}; Published: {meta.datePublished}")

## 2. Data Overview

Review available record sets, fields (columns), and their `@id`s in the dataset schema.

We use the `dataset.record_sets` property to inspect all record sets and list their associated fields.

All objects are referenced strictly by their `@id`s, following Croissant conventions.

In [ ]:
# List all available record sets and their fields by @id.
record_sets = list(dataset.record_sets.values())

if not record_sets:
    print("No record sets detected in the dataset metadata.")
else:
    print("Record sets found:")
    for rset in record_sets:
        print(f"\nRecord Set: {rset.id}")
        print(f"  name: {rset.name}")
        print("  Fields (by @id):")
        for field in rset.fields:
            print(f"    - {field.id} ({field.name}, type: {field.data_type})")

# For use in extraction below, save all record set @id values
record_set_ids = [rset.id for rset in record_sets]

## 3. Data Extraction

We load data from each available record set using its `@id`, producing a Pandas DataFrame for each.

All operations reference record sets and fields strictly by their `@id`.

In [ ]:
# Extract all available record sets as DataFrames (if any found)
dataframes = {}
for record_set_id in record_set_ids:
    # Each record is a dict keyed by field @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame.from_records(records)
        print(f"Loaded {len(dataframes[record_set_id])} records from {record_set_id}")
    else:
        print(f"Record set {record_set_id} has no records.")

if dataframes:
    # Print available columns for the first record set with data
    first_rsid = list(dataframes)[0]
    print(f"\nColumns in first record set ({first_rsid}):\n", dataframes[first_rsid].columns.tolist())
    display(dataframes[first_rsid].head())
else:
    print("No records could be loaded from any record sets.")


## 4. Exploratory Data Analysis (EDA)

Apply example data processing: filtering records by numeric field, normalizing values, grouping by a categorical field.

_All field references are by field `@id`._

In [ ]:
# Select a record set and analyze a numeric field
import numpy as np

# Safety check: run EDA only if we have extracted data
if dataframes:
    # Example: use first available record set
    record_set_id = list(dataframes)[0]
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")

    # Identify numeric fields by data_type=='schema:Float' or schema:Integer from metadata
    rs_metadata = dataset.record_sets[record_set_id]
    numeric_fields = [f.id for f in rs_metadata.fields if f.data_type in ['schema:Float', 'schema:Integer', 'schema:Number']]

    if numeric_fields:
        numeric_field = numeric_fields[0]  # Select first numeric field for demo
        print(f"Analyzing numeric field: {numeric_field}")

        if numeric_field in df.columns:
            # Attempt conversion to numeric
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
            threshold = df[numeric_field].dropna().mean() if df[numeric_field].notnull().any() else 0

            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows\n")

            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
                filtered_df[numeric_field].std()
            )
            print(f"Normalized {numeric_field} example:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

            # Attempt grouping by first available categorical field
            cat_fields = [f.id for f in rs_metadata.fields if f.data_type in [None, 'schema:Text', 'schema:Boolean'] and f.id != numeric_field]
            if cat_fields:
                group_field = cat_fields[0]

                if group_field in filtered_df.columns:
                    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
                    print(f"\nGrouped mean of {numeric_field} by {group_field}:")
                    display(grouped_df.head())
                else:
                    print(f"Field {group_field} not present in DataFrame columns.")
            else:
                print("No categorical fields found for grouping.")
        else:
            print(f"Field {numeric_field} not present in DataFrame columns.")
    else:
        print("No numeric fields available for EDA in this record set.")
else:
    print("No data loaded in dataframes; skipping EDA.")

## 5. Visualization

Visualize the numeric field distribution, and relationships to a categorical grouping (if applicable).

_All fields referenced below are by their Croissant `@id`s as above._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_id = list(dataframes)[0]
    df = dataframes[record_set_id]

    # Use previous field selections if available
    try:
        numeric_field
    except NameError:
        # Try to pick one:
        rs_metadata = dataset.record_sets[record_set_id]
        numeric_fields = [f.id for f in rs_metadata.fields if f.data_type in ['schema:Float', 'schema:Integer', 'schema:Number']]
        numeric_field = numeric_fields[0] if numeric_fields else None

    if numeric_field and numeric_field in df.columns:
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()

        # If grouping field exists, make a boxplot:
        if 'group_field' in locals() and group_field and group_field in df.columns:
            plt.figure(figsize=(10,5))
            sns.boxplot(data=df, x=group_field, y=numeric_field)
            plt.title(f"{numeric_field} by {group_field}")
            plt.show()
    else:
        print("No suitable numeric field found for visualization.")
else:
    print("No data loaded in dataframes; skipping visualization.")

## 6. Conclusion

This notebook demonstrated how to access metadata, enumerate and extract tabular data, perform simple data processing, and visualize distributions for the FAIR² Croissant dataset on rangeland management.

The exercise applied all field and record set references strictly using Croissant `@id` fields, ensuring robust, schema-driven access.

> For further analysis or machine learning tasks, continue exploring additional record sets and fields using their `@id` values as demonstrated above.